In [ ]:

!pip install lightgbm scikit-learn pandas numpy


D:\software\Python\PyCharm\PyCharm 20250201\PyCharm 2025.2.0.1\plugins\python-ce\helpers\pycharm_display\datalore\display\supported_data_type.py:6: UserWarning: The NumPy module was reloaded (imported a second time). This can in some cases result in small but subtle issues and is discouraged.
  import numpy


In [ ]:

import pandas as pd

train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/train.csv'
test_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/test.csv'

train_df = pd.read_csv(train_data_path)
test_df = pd.read_csv(test_data_path)

print(train_df.head())
print(test_df.head())


      id  no_of_adults  ...  no_of_special_requests  booking_status
0  15559             2  ...                       2               0
1  32783             2  ...                       1               0
2  11797             3  ...                       0               1
3  39750             2  ...                       1               1
4  28711             2  ...                       0               1

[5 rows x 19 columns]
      id  no_of_adults  ...  no_of_special_requests  booking_status
0   8768             2  ...                       1               0
1  38340             2  ...                       0               1
2   7104             2  ...                       0               0
3  36898             2  ...                       3               0
4   9747             2  ...                       1               0

[5 rows x 19 columns]


In [ ]:


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 假设我们已经知道哪些列是分类列
# 这里假设 'booking_status' 是我们的目标列
X_train = train_df.drop('booking_status', axis=1)
y_train = train_df['booking_status']

# 对分类列进行编码
for col in X_train.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])

# 划分验证集
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print(X_train.head())
print(y_train.head())



          id  no_of_adults  ...  avg_price_per_room  no_of_special_requests
28585  30791             1  ...               78.30                       2
11854   7977             2  ...              101.33                       0
25872   4409             2  ...              102.30                       2
31757  27117             2  ...               88.60                       1
26401  40381             2  ...               72.25                       0

[5 rows x 18 columns]
28585    1
11854    1
25872    1
31757    0
26401    0
Name: booking_status, dtype: int64


In [ ]:



import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# 创建LightGBM数据集
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# 设置模型参数
params = {
    'objective': 'binary',
    'metric': 'auc',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9
}

# 训练模型
model = lgb.train(params, train_data, valid_sets=[val_data], num_boost_round=100, early_stopping_rounds=20)

print('模型训练完成')



---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
Cell In[5], line 24
     22 pass
     23 print('##active_line18##')
---> 24 model = lgb.train(params, train_data, valid_sets=[val_data], num_boost_round=100, early_stopping_rounds=20)
     25 print('##active_line19##')
     26 pass

TypeError: train() got an unexpected keyword argument 'early_stopping_rounds'

In [ ]:



import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# 创建LightGBM数据集
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# 设置模型参数
params = {
    'objective': 'binary',
    'metric': 'auc',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9
}

# 训练模型
model = lgb.train(params, train_data, valid_sets=[val_data], num_boost_round=100, callbacks=[lgb.early_stopping(stopping_rounds=20)])

print('模型训练完成')




[LightGBM] [Info] Number of positive: 10568, number of negative: 16371
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001011 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 904
[LightGBM] [Info] Number of data points in the train set: 26939, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.392294 -> initscore=-0.437681
[LightGBM] [Info] Start training from score -0.437681
Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.885031
模型训练完成


In [ ]:


# 对测试数据进行预处理
X_test = test_df.drop('booking_status', axis=1)

# 对分类列进行编码
for col in X_test.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    if col in X_train.keys():
        X_test[col] = le.transform(X_test[col])
    else:
        X_test[col] = 0  # 如果训练集中没有该列，假设该列的编码为0

# 进行预测
predictions = model.predict(X_test, num_iteration=model.best_iteration)
predictions_binary = (predictions > 0.5).astype(int)

print(predictions_binary[:10])




[0 1 1 0 0 0 1 0 1 1]


In [ ]:



# 保存预测结果
import os

# 创建保存结果的目录（如果不存在）
output_dir = 'predictions'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, 'predictions.csv')

test_df['predicted_booking_status'] = predictions_binary
test_df[['id', 'predicted_booking_status']].to_csv(output_path, index=False)

print(f'预测结果已保存到 {output_path}')





预测结果已保存到 predictions\predictions.csv


In [ ]:



# 计算AUC-ROC
y_test = test_df['booking_status']
roc_auc = roc_auc_score(y_test, predictions)

print(f'模型在测试集上的AUC-ROC: {roc_auc:.4f}')




模型在测试集上的AUC-ROC: 0.8925
